# 3장 2강: RNN과 LSTM — 실습문제

## 실습 목표

- 순차 데이터에서 시간 순서가 중요한 이유를 설명합니다.
- RNN이 현재 입력과 이전 은닉 상태를 함께 사용함을 계산으로 확인합니다.
- 장기 의존성 문제와 LSTM의 셀 상태·게이트의 역할을 이해합니다.
- RNN과 LSTM의 공통점·차이점 및 활용 분야를 설명합니다.

## 실습 환경 / 데이터

- Python, pandas, NumPy를 사용합니다. PyTorch나 TensorFlow는 설치하지 않아도 됩니다.
- 기존 **`jena_climate_2009_2016.csv`**를 노트북과 같은 폴더에 두고, 그 폴더를 VS Code에서 열어 실행하세요. pandas와 NumPy가 설치된 Python 환경을 커널로 선택하세요.
- 원본은 420,551행, 15열입니다. 이번에는 **처음 12행**의 시간 `Date Time`과 기온 `T (degC)`를 사용합니다. 선택 구간의 관측 간격은 10분입니다.
- 처음 6개 관측값은 실습, 다음 6개는 과제에 사용합니다. 순서를 섞지 않습니다.
- 계산을 읽기 쉽게 입력을 **기온 / 10**으로 정합니다. 이는 이 예시의 고정 배율 조정이며, 평균·표준편차를 이용한 표준화는 아닙니다. 원본 기온은 변경하지 않습니다.
- 준비 코드부터 위에서 아래로 실행하세요. `round(6)`은 출력만 반올림하며 계산은 원래 값으로 진행합니다.

## 이번 강의의 범위

제공된 **3장 2강 목차: 순차 데이터와 RNN의 동작 원리, 장기 의존성 문제와 LSTM, RNN·LSTM의 특징과 활용 분야**를 기준으로 구성했습니다. 강의 본문은 별도로 제공되지 않았으므로 아래 가중치·편향과 계산 예시는 동작 원리를 보여주기 위해 정한 값입니다.

입력과 상태를 숫자 하나씩 사용하는 작은 RNN·LSTM의 **순전파**를 관찰합니다. 미분, 함수 작성, 전체 데이터 학습, 예측 성능 비교, 하이퍼파라미터 튜닝은 과제가 아닙니다. 수식과 계산 함수는 제공하며, 학생은 실행과 결과 해석에 집중합니다.

이번에 구하는 `h`와 `c`는 신경망 내부 상태입니다. **기온 예측값이나 정답 확률이 아닙니다.** 실제 기온을 예측하려면 상태를 예측 출력으로 연결하고 정답 데이터로 학습하는 과정이 추가로 필요합니다.

## 실습 구성

| 구분 | 문제 | 할 일 |
|---|---|---|
| 실습 1 | 1-1 | RNN 상태를 이어받는 계산과 매번 초기화하는 계산 비교 |
| 실습 2 | 2-1 | LSTM의 게이트·셀 상태·은닉 상태 확인 |
| 실습 2 | 2-2 | 이전 기억의 유지 비율과 장기 의존성 이해 |
| 기초 과제 | 3-1 | 다음 6개 기온에 같은 RNN·LSTM 함수 적용 |

**실습 2개 + 기초 과제 1개**이며, 마지막에는 **실습 마무리 복습 질문 6개**가 있습니다. 과제는 이미 사용한 함수와 계산을 다른 구간에 그대로 적용하는 수준입니다.

## 시작 전에 읽을 용어

| 용어 | 쉽게 풀어쓴 뜻 |
|---|---|
| 순차 데이터 | 값의 순서가 의미를 가지는 데이터. 예: 시간별 기온 |
| 시점 t | 현재 몇 번째 입력을 처리하는지 나타내는 번호 |
| RNN | 이전 상태를 다음 시점으로 전달하며 순서대로 처리하는 신경망 |
| 은닉 상태 h | 지금까지 입력한 내용을 계산해 담는 내부 상태 |
| 장기 의존성 | 멀리 떨어진 과거 정보가 현재 판단에도 필요한 관계 |
| 장기 의존성 문제 | 중요한 과거 정보를 오랜 단계에 걸쳐 활용하도록 학습하기 어려운 문제 |
| LSTM | 셀 상태와 게이트를 사용해 기억의 유지·추가·노출을 조절하는 RNN의 한 종류 |
| 셀 상태 c | 기억을 이어가기 위한 LSTM 내부의 상태 |
| 게이트 | 정보를 얼마나 통과시킬지 조절하는 값 |
| 망각 게이트 f | 이전 셀 상태를 얼마나 남길지 정함 |
| 입력 게이트 i | 새 후보 정보를 얼마나 더할지 정함 |
| 후보 정보 g | 현재 입력과 이전 은닉 상태로 계산한 새로운 기억 후보 |
| 출력 게이트 o | 셀 상태를 변환한 값을 은닉 상태에 얼마나 드러낼지 정함 |
| Sigmoid | 입력을 0과 1 사이의 값으로 바꾸는 함수. 여기서는 게이트 계산에 사용 |
| Tanh | 입력을 −1과 1 사이의 값으로 바꾸는 함수. 여기서는 상태·후보 계산에 사용 |

`t−1`은 바로 이전 시점입니다. 첫 입력 직전의 상태를 `h0`, `c0`로 씁니다. 이번 기본 계산은 모두 0에서 시작합니다. 실제 모델의 상태는 여러 숫자를 가진 벡터일 수 있지만 여기서는 하나의 숫자로 계산합니다.

### 왜 순서를 배우나요?

기온이 `10 → 12 → 14℃`로 관측된 경우와 `14 → 12 → 10℃`로 관측된 경우를 생각해 보세요. 세 숫자와 평균은 같지만, 앞의 흐름은 상승이고 뒤의 흐름은 하락입니다. **값뿐 아니라 어떤 순서로 관측되었는지도 판단에 필요할 수 있습니다.** 순차 데이터 중 시간에 따라 기록한 데이터를 시계열 데이터라고 합니다. 이번 Jena 기온 데이터가 그 예입니다.

RNN은 ‘현재 값’을 읽을 때 ‘이전까지 계산해 둔 상태’를 함께 사용합니다. RNN은 Recurrent Neural Network(순환 신경망), LSTM은 Long Short-Term Memory의 약자입니다. LSTM도 RNN 계열에 속하며, 이 강의에서 둘을 비교할 때의 ‘RNN’은 가장 기본적인 Tanh RNN을 가리킵니다.

내부 상태를 ‘메모’에 비유할 수 있습니다. 새 관측을 읽을 때 이전 메모를 참고하여 메모를 다시 쓰는 방식입니다. 다만 실제 상태는 문장이 아니라 **계산된 숫자**이고, 과거 관측을 원래 모습 그대로 모두 저장하는 목록도 아닙니다. 이번에는 설명을 위해 상태 하나를 숫자 하나로 나타냅니다.

### 기호를 처음 읽는 학생을 위한 안내

| 표시 | 읽는 방법 | 구분할 점 |
|---|---|---|
| $t$ | 현재 시점 번호 | 이번 구간의 1번째, 2번째 입력을 구분 |
| $t-1$ | 바로 이전 시점 | t가 2이면 t−1은 1 |
| $x_t$ | t번째 입력값 | 아래 t는 제곱이 아니라 순서 표시 |
| $h_{t-1}$ | 직전 시점의 은닉 상태 | 이전 입력들로 계산한 내부 숫자 |
| $h_t$ | 이번에 새로 만든 은닉 상태 | 다음 시점에서 이전 상태로 사용 |
| $c_{t-1},c_t$ | 이전 셀 상태, 새 셀 상태 | LSTM에서 기억을 이어 가는 별도 상태 |
| $h_0,c_0$ | 첫 입력을 읽기 전 상태 | 이번 예시는 0으로 시작 |
| $z_t,s_t$ | 활성화 함수에 넣기 전 가중합 | 입력·이전 상태에 가중치를 곱해 더한 값 |
| $\sigma$ | ‘시그마’, 여기서는 Sigmoid 함수 | 합계를 뜻하는 큰 기호 Σ와 구분 |
| $\tanh$ | ‘하이퍼볼릭 탄젠트’, Tanh 함수 | 숫자를 −1과 1 사이로 변환 |
| $\approx$ | 대략 같다 | 소수점 아래를 반올림하여 표시 |

수식에서 $0.5x_t$는 `0.5 × x_t`처럼 곱하기 기호를 생략한 표기입니다. $f_t c_{t-1}$도 두 숫자를 곱한다는 뜻입니다. Tanh와 Sigmoid의 함수값은 제공 코드로 확인하면 됩니다. 두 함수를 손으로 근사하거나 미분할 필요는 없습니다.

**시점, 층, 학습을 구분하세요.** 기온 6개를 순서대로 처리하는 것은 같은 계산을 시간 방향으로 여섯 번 사용하는 것입니다. 은닉층을 여섯 개 쌓거나 학습을 여섯 번 했다는 뜻이 아닙니다. 이번에는 가중치·편향을 고정하고 상태만 바뀌는 순전파를 관찰합니다.

### 주제별 학습 위치

| 강의 주제 | 확인할 문제 | 학습 후 설명할 내용 |
|---|---|---|
| 순차 데이터와 RNN의 동작 원리 | 실습 1-1, 과제 3-1 | 시간 순서, 현재 입력과 이전 상태, 상태 전달과 초기화 |
| 장기 의존성 문제와 LSTM | 실습 2-1·2-2 | 셀 상태·게이트, 과거 정보의 유지, 긴 관계 학습의 어려움 |
| RNN·LSTM의 특징과 활용 분야 | 실습 2-2 Q4, 과제 3-1 Q4 | 공통점·차이점, 기상·설비 데이터에서 순서가 필요한 이유 |

과제는 실습에서 사용한 함수를 다음 기온 구간에 적용하는 기초 수준입니다.


## 실습 준비 — 제공 코드 세 셀을 그대로 실행하세요

첫 셀은 데이터를 읽고, 두 번째는 RNN 함수, 세 번째는 LSTM 함수를 준비합니다. **제공 함수의 내부를 새로 작성하거나 수식을 암기할 필요는 없습니다.**

### RNN 계산 규칙

$$h_t=\tanh(0.5x_t+0.5h_{t-1}),\qquad h_0=0$$

현재 입력에 0.5를 곱하고, 이전 은닉 상태에도 0.5를 곱해 더한 다음 Tanh를 적용합니다. 편향은 0입니다. 모든 시점에서 **같은 가중치**를 사용합니다.

- `run_rnn(values)`는 이전 상태를 이어받습니다.
- `run_rnn(values, reset_each=True)`는 매 시점에 이전 상태를 0으로 만드는 비교용 계산입니다. 정상적인 상태 전달을 일부러 끊은 경우입니다.

### LSTM 계산 규칙

기본 LSTM의 게이트 구조를 숫자 하나로 줄여 계산합니다. 이 예시에서는 각 계산의 입력 가중치를 0.5, 이전 은닉 상태 가중치를 0.2로 동일하게 정했으므로 공통 가중합 `s`를 사용합니다. 망각 게이트 편향만 1이고 나머지 편향은 0입니다. **일반적인 LSTM에서는 각 게이트와 후보 계산의 가중치·편향이 별도로 학습됩니다.**

$$s_t=0.5x_t+0.2h_{t-1}$$
$$f_t=\sigma(s_t+1),\quad i_t=\sigma(s_t),\quad g_t=\tanh(s_t),\quad o_t=\sigma(s_t)$$
$$c_t=f_t c_{t-1}+i_t g_t,\qquad h_t=o_t\tanh(c_t)$$

`σ`는 Sigmoid입니다. 계산 순서는 **현재 입력·이전 h → 게이트와 후보 → 새 c → 새 h**입니다. `c`를 갱신할 때 이전 기억을 남기는 항과 새 정보를 더하는 항을 합칩니다. 후보 g는 게이트가 아니며 음수일 수도 있습니다.

- `run_lstm(values)`는 처음에 h와 c를 각각 0으로 만든 뒤, 한 구간 안에서는 두 상태를 계속 이어받습니다.
- 반환 표의 `keep=f×c_prev`는 남긴 이전 기억, `add=i×g`는 더할 새 정보입니다.
- **함수 호출 한 번은 독립된 구간 하나**입니다. 과제에서는 실습의 마지막 상태를 이어받지 않고 새로 시작합니다.

`iloc[:6]`은 처음 6행이고 `iloc[6:12]`는 7번째부터 12번째 행입니다. `table.iloc[-1]`은 결과 표의 마지막 행입니다.

### 수식과 코드의 연결

| 코드 | 말로 읽기 |
|---|---|
| `h = 0.0` | 이번 구간의 시작 상태를 0으로 준비 |
| `h_prev = h` | 새 값을 계산하기 전에 직전 상태를 보관 |
| `z = 0.5 * value + 0.5 * h_prev` | 현재 입력의 절반과 이전 상태의 절반을 더함 |
| `h = np.tanh(z)` | 가중합에 Tanh를 적용하고 새 h로 저장 |
| `keep = f * c_prev` | 이전 셀 상태에 남길 비율을 곱함 |
| `add = i * g` | 후보 정보에 반영할 비율을 곱함 |
| `c = keep + add` | 남긴 이전 값과 새로 반영할 값을 합침 |
| `h = o * np.tanh(c)` | 새 셀 상태를 변환한 뒤 출력 게이트 비율을 곱함 |

코드의 `=`는 수학적 등식을 증명하라는 뜻이 아니라, 오른쪽 계산 결과를 왼쪽 이름에 저장하라는 뜻입니다. `c = f * c`라면 오른쪽의 **이전 c**로 계산한 결과를 왼쪽의 **새 c**로 저장합니다.

`run_rnn(x)`처럼 함수를 호출하면 준비된 계산을 수행하고 결과 표를 반환합니다. `table["h"]`는 표의 h열, `iloc[0]`은 첫 행, `iloc[-1]`은 마지막 행입니다. Python 배열 위치는 0부터 시작하지만, 표의 t열은 사람이 읽기 편하게 1부터 붙인 시점 번호입니다.

`reset_each=False`는 매번 초기화하지 않는 기본 설정이고, `True`를 주면 매 시점의 이전 h를 0으로 만듭니다. 단어 `reset`은 초기화, `each`는 매번이라는 뜻입니다. 함수 안의 `for`는 각 입력에 대해 계산을 반복합니다.


### 코드를 처음 읽을 때 확인할 기호

| 표현 | 쉬운 뜻 | 이번 실습에서의 사용 |
|---|---|---|
| `# ...` | 사람이 읽는 주석이며 계산하지 않습니다. | 각 줄이 하는 일을 설명합니다. |
| `이름 = 계산` | 오른쪽 계산 결과를 왼쪽 이름에 저장합니다. | `h = np.tanh(z)`는 새 상태를 h에 저장합니다. |
| `def 함수이름(...):` | 나중에 실행할 계산 방법을 정의합니다. | 이 셀만 실행하면 함수가 준비되고, 실제 계산은 `run_rnn(x)`처럼 호출할 때 시작합니다. |
| `for ...:` | 여러 입력에 대해 같은 일을 반복합니다. | 기온을 순서대로 하나씩 읽습니다. |
| `if 조건:` | 조건이 맞을 때만 아래 내용을 실행합니다. | 특정 단계에서만 결과를 기록합니다. |
| `True`, `False` | 참과 거짓을 나타냅니다. | 매번 초기화 옵션을 켜거나 끕니다. |
| `[]` | 여러 값을 담는 목록 또는 필요한 부분을 고르는 표시입니다. | `rows = []`는 기록을 담을 빈 목록입니다. |
| `{"이름": 값}` | 이름과 값을 짝지어 묶습니다. 이를 딕셔너리라고 합니다. | `{"t": t, "h": h}`는 한 시점의 번호와 상태를 기록합니다. |
| `.append(값)` | 목록 맨 뒤에 값을 추가합니다. | 한 시점의 기록을 rows에 넣습니다. |
| `return 결과` | 함수의 결과를 호출한 곳에 돌려줍니다. | 결과 표를 돌려주며, 화면에 보이게 하려면 `print`를 사용합니다. |

**들여쓰기는 실행 범위입니다.** `for` 아래에서 안쪽으로 들어간 줄은 매 시점 반복됩니다. 반복문 바깥에 있는 `h = 0.0`은 함수 호출 때 한 번만 실행됩니다. 이 위치 차이가 ‘처음 한 번 초기화’와 ‘매번 초기화’를 구분하는 데 중요합니다.

**읽는 순서:** 데이터 준비 → 함수 정의 → 문제 코드 실행 → 출력 표에서 첫 행과 둘째 행 비교 → 해석 질문 작성 → 실습 마무리입니다. 제공 함수를 외우거나 새로 구현할 필요는 없습니다. 막히는 줄은 정답 파일의 준비 코드 해설에서 확인하세요.


In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv("jena_climate_2009_2016.csv", nrows=12)
times = pd.to_datetime(df["Date Time"], format="%d.%m.%Y %H:%M:%S")
# 앞 6개 기온을 배열로 만들고 각각 10으로 나눕니다.
x = df["T (degC)"].iloc[:6].to_numpy(dtype=float) / 10.0
# 다음 6개 기온을 과제 입력으로 따로 준비합니다.
x_home = df["T (degC)"].iloc[6:12].to_numpy(dtype=float) / 10.0

# 읽은 표의 행·열 개수를 확인합니다.
print("읽은 데이터 크기:", df.shape)
# 선택 구간의 날짜와 기온만 표시합니다.
print(df[["Date Time", "T (degC)"]].to_string(index=False))
# 이웃한 시각 차이를 초로 바꾼 뒤 60으로 나누어 분 단위로 확인합니다.
print("선택 구간 간격(분):", times.diff().dropna().dt.total_seconds().unique() / 60)
print("선택 기온의 결측값 수:", df["T (degC)"].isna().sum())
print("실습 입력(기온/10):", x)
print("과제 입력(기온/10):", x_home)


In [ ]:
# 함수 정의입니다. reset_each를 생략하면 False로 계산합니다.
def run_rnn(values, reset_each=False):
    # 호출된 구간의 시작 상태입니다. 반복문 밖이므로 처음 한 번만 초기화합니다.
    h = 0.0
    rows = []

    for t, value in enumerate(values, start=1):
        # True이면 0을 쓰고, False이면 이전 반복에서 구한 h를 사용합니다.
        h_prev = 0.0 if reset_each else h
        # 현재 입력 항과 이전 상태 항을 더합니다. 가중치는 모든 시점에서 같습니다.
        z = 0.5 * value + 0.5 * h_prev
        # Tanh를 적용한 새 상태를 저장합니다. 다음 반복에서 이어받습니다.
        h = np.tanh(z)
        # 계산을 설명하기 위해 이번 시점의 값을 한 행으로 기록합니다.
        rows.append({"t": t, "x": value, "h_prev": h_prev, "z": z, "h": h})

    return pd.DataFrame(rows)


print("RNN 함수 준비 완료")


In [ ]:
# 0과 1 사이의 게이트 값을 계산할 함수를 정의합니다.
def sigmoid(value):
    # exp는 지수 함수입니다. 1을 괄호 안 분모로 나눈 값을 반환합니다.
    return 1.0 / (1.0 + np.exp(-value))


# 입력 배열 한 구간에 대해 LSTM 상태를 차례로 계산합니다.
def run_lstm(values):
    # 첫 입력 전 은닉 상태를 0으로 준비합니다.
    h = 0.0
    # 첫 입력 전 셀 상태도 0으로 준비합니다.
    c = 0.0

    rows = []

    for t, value in enumerate(values, start=1):
        # 새 상태로 덮어쓰기 전에 이전 h를 보관합니다.
        h_prev = h
        # 새 상태로 덮어쓰기 전에 이전 c를 보관합니다.
        c_prev = c
        # 이번 예시에서 게이트와 후보가 사용할 공통 가중합입니다.
        s = 0.5 * value + 0.2 * h_prev
        # 망각 게이트: 편향 1을 더하고, 이전 c를 남길 비율을 계산합니다.
        f = sigmoid(s + 1.0)
        # 입력 게이트: 새 후보 g를 반영할 비율입니다.
        i = sigmoid(s)
        # Tanh로 새 후보를 만듭니다. 게이트가 아니므로 음수도 가능합니다.
        g = np.tanh(s)
        # 출력 게이트: 변환한 c를 h에 드러낼 비율입니다.
        o = sigmoid(s)
        # 이전 셀 상태에서 남기는 항입니다.
        keep = f * c_prev
        # 새 후보에서 반영할 항입니다. 음수가 될 수도 있습니다.
        add = i * g
        # 남긴 항과 새 항을 더해 새 셀 상태를 만듭니다.
        c = keep + add
        # 방금 만든 새 c에 Tanh를 적용하고 o를 곱해 새 h를 만듭니다.
        h = o * np.tanh(c)

        rows.append({"t": t, "x": value, "h_prev": h_prev,
                     "f": f, "i": i, "g": g, "o": o,
                     "c_prev": c_prev, "keep": keep, "add": add,
                     "c": c, "h": h})

    return pd.DataFrame(rows)


print("LSTM 함수 준비 완료")


---

## 실습 1. 순차 데이터와 RNN의 동작 원리


### 문제 1-1. 이전 상태를 전달하면 무엇이 달라질까요?

#### 문제 설명

첫 6개 기온을 시간 순서대로 처리합니다. 정상적으로 이전 상태를 전달하는 경우와 매번 초기화하는 경우를 비교하세요. 입력과 가중치는 두 경우에 동일합니다.

#### 요구사항

1. `run_rnn(x)`로 `rnn_table`을 만드세요.
2. `run_rnn(x, reset_each=True)`로 `reset_table`을 만드세요.
3. 두 표를 출력하고 마지막 은닉 상태를 비교하세요.
4. 상태를 이어받는 경우의 첫 번째와 두 번째 h를 수식에 대입해 계산하세요. Tanh 값은 `np.tanh()`로 확인해도 됩니다.

#### 작성 도움말

`print(rnn_table.round(6).to_string(index=False))`로 출력합니다. 마지막 상태는 `rnn_table["h"].iloc[-1]`입니다. 첫 입력은 −0.802, 두 번째 입력은 −0.841입니다.

#### 해석 질문

**Q1.** 기온 데이터에서 시간 순서를 유지해야 하는 이유는 무엇인가요? 이번 입력 x와 원래 기온의 관계는 무엇인가요?

**Q2.** 상태를 이어받는 RNN의 h1과 h2는 각각 얼마인가요? 두 번째 계산에 첫 번째 입력의 정보가 어떻게 들어가나요?

**Q3.** 두 경우의 마지막 h는 각각 얼마인가요? 첫 번째 h는 같고 두 번째부터 달라지는 이유를 설명하세요.

**Q4.** h가 음수이면 잘못 계산한 것인가요? 마지막 h에 10을 곱하면 다음 기온 예측값이 되나요?

#### 제출 결과

두 결과 표, 첫 두 시점의 대입 계산, 마지막 상태 비교, Q1~Q4 답변.


In [ ]:
# 같은 x로 두 경우를 각각 실행하세요.
# rnn_table = ...
# reset_table = ...

# 두 표와 마지막 h를 출력하세요.


#### 답변 작성란

**수식 및 계산 과정**

- 작성: 

**Q1. 기온 데이터에서 시간 순서를 유지해야 하는 이유는 무엇인가요? 이번 입력 x와 원래 기온의 관계는 무엇인가요?**

- 답변: 

**Q2. 상태를 이어받는 RNN의 h1과 h2는 각각 얼마인가요? 두 번째 계산에 첫 번째 입력의 정보가 어떻게 들어가나요?**

- 답변: 

**Q3. 두 경우의 마지막 h는 각각 얼마인가요? 첫 번째 h는 같고 두 번째부터 달라지는 이유를 설명하세요.**

- 답변: 

**Q4. h가 음수이면 잘못 계산한 것인가요? 마지막 h에 10을 곱하면 다음 기온 예측값이 되나요?**

- 답변:


---

## 실습 2. LSTM과 장기 의존성


### 문제 2-1. LSTM은 기억을 어떻게 남기고 더할까요?

#### 문제 설명

같은 실습 입력 x를 LSTM 함수에 넣습니다. 게이트·셀 상태·은닉 상태를 확인하고 첫 두 시점에서 이전 기억과 새 정보가 어떻게 합쳐지는지 해석하세요.

#### 요구사항

1. `run_lstm(x)`로 `lstm_table`을 만드세요.
2. 게이트 표 `t, x, f, i, g, o`와 상태 표 `t, c_prev, keep, add, c, h`를 나누어 출력하세요.
3. 첫 시점의 f·i·g·o 값을 읽고 c1과 h1을 계산하세요. 함수가 출력한 값을 사용해도 됩니다.
4. 두 번째 시점의 `keep + add = c`를 확인하세요.
5. 마지막 c와 h를 출력하세요.

#### 작성 도움말

`lstm_table[["t", "x", "f", "i", "g", "o"]]`처럼 여러 열을 선택할 수 있습니다. `keep=f*c_prev`, `add=i*g`, `h=o*np.tanh(c)`입니다. 손계산은 반올림에 따른 작은 차이를 허용합니다.

#### 해석 질문

**Q1.** 망각·입력·출력 게이트와 후보 g의 역할은 각각 무엇인가요? f=0.8은 이전 기억의 80%를 버린다는 뜻인가요?

**Q2.** 첫 시점 c1과 h1은 각각 얼마인가요? 첫 시점의 keep이 0인 이유는 무엇인가요?

**Q3.** 두 번째 시점의 keep, add, c는 각각 얼마인가요? 마지막 c와 h도 쓰고, 둘이 같은 값이어야 하는지 설명하세요.

**Q4.** 출력 게이트 o는 미래 기온을 직접 출력하는 층인가요? 이번 예시에서 i와 o가 같은 이유는 무엇인가요?

#### 제출 결과

게이트·상태 표, 첫 두 시점 계산, 마지막 c·h, Q1~Q4 답변.


In [ ]:
# 제공 함수로 LSTM을 실행하세요.
# lstm_table = ...

# 게이트 표와 상태 표를 나누어 출력하세요.
# 마지막 c와 h를 출력하세요.


#### 답변 작성란

**수식 및 계산 과정**

- 작성: 

**Q1. 망각·입력·출력 게이트와 후보 g의 역할은 각각 무엇인가요? f=0.8은 이전 기억의 80%를 버린다는 뜻인가요?**

- 답변: 

**Q2. 첫 시점 c1과 h1은 각각 얼마인가요? 첫 시점의 keep이 0인 이유는 무엇인가요?**

- 답변: 

**Q3. 두 번째 시점의 keep, add, c는 각각 얼마인가요? 마지막 c와 h도 쓰고, 둘이 같은 값이어야 하는지 설명하세요.**

- 답변: 

**Q4. 출력 게이트 o는 미래 기온을 직접 출력하는 층인가요? 이번 예시에서 i와 o가 같은 이유는 무엇인가요?**

- 답변:


### 문제 2-2. 멀리 떨어진 과거 기억을 유지하기

#### 문제 설명

장기 의존성은 **여러 시점 전의 중요한 정보가 현재 판단에도 필요한 관계**입니다. 예를 들어 일정 시간 전부터 시작된 온도 상승 흐름을 현재의 장비 상태 판단에 활용하는 경우입니다.

RNN은 과거 상태를 반복 변환합니다. 학습할 때도 여러 시점을 거슬러 신호가 전달되므로 기울기가 매우 작아지거나 커져 긴 관계를 학습하기 어려울 수 있습니다. LSTM은 셀 상태와 게이트를 통해 오래 유지할 정보와 새로 반영할 정보를 조절하도록 설계되었습니다.

이번에는 셀 상태 식에서 **이전 기억을 유지하는 항 하나**만 살펴봅니다. 아래는 원리를 분리한 실험이며, 직전 문제의 LSTM 전체를 그대로 실행한 결과가 아닙니다.

$$c_t=f\times c_{t-1},\qquad c_0=1$$

- 초기 기억은 1입니다.
- 새 정보 항 `i*g`는 0으로 고정합니다.
- 망각 게이트는 0.5 또는 0.9로 고정합니다.
- 12번 반복해 1·6·12단계의 c를 비교합니다. 단계 번호는 기상 관측 시각이 아니라 이 설명용 실험의 반복 횟수입니다.

#### 요구사항

1. 두 경우 모두 초기 기억 1에서 별도로 시작하세요.
2. 아래 반복문 틀을 완성해 12단계 동안 이전 c에 f를 곱하세요.
3. 1·6·12단계에서 남은 기억을 표로 출력하세요.
4. 결과를 바탕으로 LSTM의 기억 조절과 한계를 설명하세요.

#### 작성 도움말

`c = f * c`를 반복하면 됩니다. `f**12`로 마지막 값을 확인해도 됩니다. 학습률이나 예측 오차를 구하는 문제는 아닙니다.

#### 해석 질문

**Q1.** f=0.5와 f=0.9에서 1·6·12단계 후 남은 기억은 각각 얼마인가요? 어떤 경우에 더 많이 남나요?

**Q2.** 이 실험을 RNN과 LSTM의 예측 성능 비교라고 할 수 있나요? 0.5를 RNN, 0.9를 LSTM의 고정된 성질로 이해해도 되나요?

**Q3.** 장기 의존성 문제가 무엇이며 LSTM은 어떻게 완화하나요? LSTM이라면 언제나 모든 과거를 완벽히 기억하나요?

**Q4.** 기본 RNN과 LSTM의 공통점·차이점을 정리하고, 활용 분야 두 가지를 순서 정보가 필요한 이유와 함께 설명하세요.

#### 제출 결과

기억 유지 결과 표, Q1~Q4 답변. 기울기 미분이나 실제 모델 학습은 요구하지 않습니다.


In [ ]:
# rows = []
# for f in [0.5, 0.9]:
#     c = 1.0
#     for step in range(1, 13):
#         c = ...
#         if step in [1, 6, 12]:
#             rows.append({"f": f, "step": step, "c": c})
# memory_table = pd.DataFrame(rows)
# print(memory_table.round(6).to_string(index=False))


#### 답변 작성란

**수식 및 계산 과정**

- 작성: 

**Q1. f=0.5와 f=0.9에서 1·6·12단계 후 남은 기억은 각각 얼마인가요? 어떤 경우에 더 많이 남나요?**

- 답변: 

**Q2. 이 실험을 RNN과 LSTM의 예측 성능 비교라고 할 수 있나요? 0.5를 RNN, 0.9를 LSTM의 고정된 성질로 이해해도 되나요?**

- 답변: 

**Q3. 장기 의존성 문제가 무엇이며 LSTM은 어떻게 완화하나요? LSTM이라면 언제나 모든 과거를 완벽히 기억하나요?**

- 답변: 

**Q4. 기본 RNN과 LSTM의 공통점·차이점을 정리하고, 활용 분야 두 가지를 순서 정보가 필요한 이유와 함께 설명하세요.**

- 답변:


---

## 기초 과제. 다음 기온 구간에서 상태 전달 확인하기


### 문제 3-1. 같은 RNN·LSTM을 다른 구간에 적용하기

#### 문제 설명

준비 코드의 `x_home`은 7번째부터 12번째 기온을 10으로 나눈 입력입니다. 실습과 동일한 RNN·LSTM 함수를 사용하세요. **두 함수 모두 새 구간으로 시작하므로 처음 상태는 0**입니다. 실습의 마지막 상태를 넘기지 않습니다.

#### 요구사항

1. `run_rnn(x_home)`로 `home_rnn`을 만드세요. 매 시점 초기화 옵션은 사용하지 않습니다.
2. `run_lstm(x_home)`로 `home_lstm`을 만드세요.
3. RNN의 t·x·h, LSTM의 t·x·c·h를 출력하세요.
4. 첫 RNN 상태와 첫 LSTM 셀 상태를 수식에 대입해 확인하세요. 제공 함수의 게이트·후보 값을 참고해도 됩니다.
5. 마지막 RNN h, LSTM c·h를 출력하고 의미를 설명하세요.

#### 작성 도움말

과제의 첫 입력은 −0.762입니다. 첫 RNN은 `tanh(0.5*x_home[0])`이며, 첫 LSTM은 c0=0이므로 `c1=i1*g1`입니다. 이전 상태를 계속 전달하는 부분은 제공 함수가 처리합니다.

#### 해석 질문

**Q1.** 첫 RNN h와 첫 LSTM c는 각각 얼마인가요? 대입 과정을 쓰세요.

**Q2.** 마지막 RNN h와 LSTM c·h는 각각 얼마인가요? 실습과 같은 함수인데 숫자가 달라지는 이유는 무엇인가요?

**Q3.** 처음에 상태를 0으로 시작하는 것과 매 시점 0으로 초기화하는 것은 어떻게 다른가요?

**Q4.** 마지막 h의 크기만 보고 RNN과 LSTM 중 예측 성능이 더 좋은 모델을 고를 수 있나요? 이유를 설명하세요.

#### 제출 결과

두 상태 표, 첫 시점 계산, 마지막 상태, Q1~Q4 답변. 별도 학습이나 심화 구현은 요구하지 않습니다.


In [ ]:
# 동일한 제공 함수에 과제 입력을 넣으세요.
# home_rnn = ...
# home_lstm = ...

# 지정한 상태 열과 마지막 상태를 출력하세요.


#### 답변 작성란

**수식 및 계산 과정**

- 작성: 

**Q1. 첫 RNN h와 첫 LSTM c는 각각 얼마인가요? 대입 과정을 쓰세요.**

- 답변: 

**Q2. 마지막 RNN h와 LSTM c·h는 각각 얼마인가요? 실습과 같은 함수인데 숫자가 달라지는 이유는 무엇인가요?**

- 답변: 

**Q3. 처음에 상태를 0으로 시작하는 것과 매 시점 0으로 초기화하는 것은 어떻게 다른가요?**

- 답변: 

**Q4. 마지막 h의 크기만 보고 RNN과 LSTM 중 예측 성능이 더 좋은 모델을 고를 수 있나요? 이유를 설명하세요.**

- 답변:


## 제출 전 확인

- 실습 2개와 기초 과제의 요구사항을 모두 수행했나요?
- 각 문제 Q1~Q4에 답했나요?
- 상태를 처음에 한 번 초기화하는 것과 매번 초기화하는 것을 구분했나요?
- 셀 상태·은닉 상태·예측값을 혼동하지 않았나요?
- 장기 의존성 실험의 의미와 한계를 설명했나요?


---

## 실습 마무리

실습 1·실습 2·기초 과제에서 배운 내용을 자신의 말로 정리하세요. 새로운 심화 문제가 아니라 **앞에서 계산하고 해석한 내용을 확인하는 복습**입니다. Tanh와 Sigmoid 값은 제공 코드 또는 앞의 계산 표를 참고해도 됩니다.

1. 순차 데이터에서 순서가 중요한 이유를 설명하세요. RNN이 이전 h를 다음 시점에 전달하는 것과 은닉층을 여러 개 쌓거나 학습을 반복하는 것은 어떻게 다른가요?

2. 상태를 이어받는 RNN에서 첫 입력 −0.802와 둘째 입력 −0.841을 처리한 h1·h2를 계산하세요. 매 시점 초기화하는 경우의 h2는 왜 다른가요?

3. LSTM의 f·i·g·o, c·h는 각각 어떤 역할을 하나요? f=0.8일 때 무엇의 80%를 남기며, 출력 게이트 o는 최종 예측 출력층과 같은가요?

4. LSTM 첫 시점에서 c0=0, f1≈0.645427, i1=o1≈0.401072, g1≈−0.380804일 때 keep·add·c1·h1을 계산하세요. 다음 시점에 전달되는 두 상태는 무엇인가요?

5. 초기 c=1에서 새 정보 없이 f를 계속 곱하면 f=0.5와 f=0.9의 12단계 결과는 각각 얼마인가요? 장기 의존성과의 관계 및 이 실험으로 판단할 수 없는 것을 설명하세요.

6. 기본 RNN과 LSTM의 공통점·차이점, 활용 분야 두 가지를 설명하세요. 과제에서 처음 두 입력이 같은데 상태가 다른 이유와 h·c만으로 성능을 비교할 수 없는 이유도 설명하세요.


### 실습 마무리 답변 작성란

**1. 순차 데이터에서 순서가 중요한 이유를 설명하세요. RNN이 이전 h를 다음 시점에 전달하는 것과 은닉층을 여러 개 쌓거나 학습을 반복하는 것은 어떻게 다른가요?**

- 답변: 

- 계산 과정 또는 예시: 

**2. 상태를 이어받는 RNN에서 첫 입력 −0.802와 둘째 입력 −0.841을 처리한 h1·h2를 계산하세요. 매 시점 초기화하는 경우의 h2는 왜 다른가요?**

- 답변: 

- 계산 과정 또는 예시: 

**3. LSTM의 f·i·g·o, c·h는 각각 어떤 역할을 하나요? f=0.8일 때 무엇의 80%를 남기며, 출력 게이트 o는 최종 예측 출력층과 같은가요?**

- 답변: 

- 계산 과정 또는 예시: 

**4. LSTM 첫 시점에서 c0=0, f1≈0.645427, i1=o1≈0.401072, g1≈−0.380804일 때 keep·add·c1·h1을 계산하세요. 다음 시점에 전달되는 두 상태는 무엇인가요?**

- 답변: 

- 계산 과정 또는 예시: 

**5. 초기 c=1에서 새 정보 없이 f를 계속 곱하면 f=0.5와 f=0.9의 12단계 결과는 각각 얼마인가요? 장기 의존성과의 관계 및 이 실험으로 판단할 수 없는 것을 설명하세요.**

- 답변: 

- 계산 과정 또는 예시: 

**6. 기본 RNN과 LSTM의 공통점·차이점, 활용 분야 두 가지를 설명하세요. 과제에서 처음 두 입력이 같은데 상태가 다른 이유와 h·c만으로 성능을 비교할 수 없는 이유도 설명하세요.**

- 답변: 

- 계산 과정 또는 예시:
